# 4. Score molecules with a trained model

Scores any SMILES with the models of a finished run, with the same code the benchmark used, so the numbers
match. Replaces the `*_inference*` notebooks' "score my own molecules" use.

It works for both methods: a `finetune` run uses its fine-tuned MAMMAL model, an `embed_lgbm` run embeds the
molecules and scores them with its LightGBM models.

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"
RUN_NAME = "lgbm_pretrained"           # a finished run folder in the experiment
MODELS = "full"                        # "full" = the model trained on all rows; "folds" = fold models averaged

SMILES = ["CCO", "c1ccccc1O", "CC(=O)Nc1ccc(O)cc1", "CN1C=NC2=C1C(=O)N(C)C(=O)N2C", "not_a_smiles"]
BATCH_SIZE = 16

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

In [ ]:
import json

RUN_DIR = EXP_DIR / RUN_NAME
assert (RUN_DIR / "run.json").exists(), f"No finished run in {RUN_DIR}"
run_info = json.loads((RUN_DIR / "run.json").read_text())
print(f"{run_info['name']}  ({run_info['method']}, {run_info['created']})")
print(f"settings: {run_info['settings']}")

## Score

A `finetune` run scores the molecules directly. An `embed_lgbm` run embeds them with its model first, then
uses its LightGBM models (`full.txt`, or every `fold_*.txt` averaged).

In [ ]:
import numpy as np

from methods import common, embeddings

if run_info["method"] == "finetune":
    folder = RUN_DIR / "models" / ("final" if MODELS == "full" else "fold_0")
    paths = [folder / "best_model"] if MODELS == "full" else sorted(RUN_DIR.glob("models/fold_*/best_model"))
    scores = []
    for path in paths:
        print(f"scoring with {path.relative_to(RUN_DIR)}")
        scores.append(common.score_with(common.load_model(path), SMILES, BATCH_SIZE))
    scores = np.mean(scores, axis=0)
else:
    import lightgbm as lgb

    config = json.loads((RUN_DIR / "config.json").read_text())
    model = common.load_model(run_info["settings"].get("encoder"))
    cleaned, valid = common.valid_smiles(SMILES)
    kept = [s for s, ok in zip(cleaned, valid) if ok]
    loader = common.make_loader(pd.DataFrame({"smiles": kept}), batch_size=BATCH_SIZE, with_label=False)
    import torch
    model = model.to(common.DEVICE).eval()
    parts = []
    with torch.no_grad():
        for batch in loader:
            for key in (common.ENCODER_INPUTS_TOKENS, common.ENCODER_INPUTS_ATTENTION_MASK):
                batch[key] = batch[key].to(common.DEVICE)
            parts.append(embeddings.embed_batch(model, batch))
    X = np.concatenate(parts)
    files = [RUN_DIR / "models" / "full.txt"] if MODELS == "full" else sorted((RUN_DIR / "models").glob("fold_*.txt"))
    print(f"scoring with {[f.name for f in files]} on {config['view']} embeddings")
    scores = np.full(len(SMILES), np.nan)
    scores[np.array(valid)] = np.mean([lgb.Booster(model_file=str(f)).predict(X) for f in files], axis=0)

In [ ]:
results = pd.DataFrame({"smiles": SMILES, "score": scores})
results.sort_values("score", ascending=False, na_position="last")

`score` is the model's probability that the molecule is a hit. A SMILES RDKit cannot read gets no
score.

To score a whole file, read it into a list of SMILES and pass that; for thousands of molecules use a
benchmark run (`run.py`), which also writes the predictions and the metrics.